## Setting up Transformer Models for Comparison

This notebook will guide you through setting up and comparing three popular transformer models (BERT, DistilBERT, and RoBERTa) on a given Excel dataset. The general steps involve:

1.  **Installing Libraries**: Install `transformers`, `pandas`, and `torch`.
2.  **Loading Data**: Load your Excel file into a pandas DataFrame.
3.  **Data Preprocessing**: Prepare the text data for each model's tokenizer.
4.  **Model Initialization**: Load pre-trained models and their respective tokenizers.
5.  **Task Definition & Fine-tuning/Prediction**: Define a specific NLP task (e.g., text classification) and either fine-tune the models or perform inference.
6.  **Performance Evaluation**: Compare the models' performance using appropriate metrics.

In [ ]:
# Install necessary libraries
!pip install transformers pandas torch

# Or if you prefer tensorflow:
# !pip install transformers pandas tensorflow

### 1. Load your Excel Data

Replace `'your_data.xlsx'` with the actual path to your Excel file. Ensure the file contains a column with text data that you want to process.

In [ ]:
import pandas as pd

# --- IMPORTANT: Upload your Excel file to Colab or provide the correct path ---
# For example, if your file is named 'my_texts.xlsx' and it's directly in your Colab files:
file_path = 'SytheticContradetect.xlsx' # <--- CHANGE THIS TO YOUR EXCEL FILE PATH

try:
    df = pd.read_excel(file_path)
    print(f"Successfully loaded data from {file_path}. Shape: {df.shape}")
    display(df.head())
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found. Please upload it to Colab or provide the correct path.")
    print("You can upload files by clicking the folder icon on the left sidebar, then the 'Upload to session storage' icon.")
    df = None # Set df to None to avoid errors in subsequent cells if file isn't found

Successfully loaded data from /content/Sythetic Contradetect V1.xlsx. Shape: (2580, 5)


,Premise,Hypothesis,Relationship,ContradictionType,Subject
0,The quantum processor developed by QTech Corp ...,The quantum processor from QTech Corp is limit...,Contradictory,Numerical,Technology
1,The new privacy update encrypts all user data ...,The new privacy update fails to encrypt any us...,Contradictory,Negation,Technology
2,The AI model was trained using a dataset consi...,The AI model was trained using diverse dataset...,Contradictory,Attribute,Technology
3,Tesla’s autopilot system was launched in 2015 ...,Tesla’s autopilot was introduced in 2022 after...,Contradictory,Temporal,Technology
4,The cloud server in Frankfurt ensures data is ...,The cloud server is located in Singapore to re...,Contradictory,Spatial,Technology


### 2. Initialize Transformer Models and Tokenizers

We will now load the pre-trained tokenizers and models for BERT, DistilBERT, and RoBERTa. For demonstration, we'll use the base versions of these models suitable for sequence classification. If your task is different (e.g., token classification, question answering), you might need to adjust the model type (e.g., `AutoModelForTokenClassification`).

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Define device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Placeholder for number of labels. This will be updated after data preprocessing.
# For now, let's keep it as 2, and we'll ensure it's adjusted after the label creation step.
# Or dynamically set it if we have 'df' available which we don't here.
# We will assume it will be correctly set by the subsequent data preprocessing step
# to the number of unique labels in the 'Relationship' column.
num_unique_labels = 3 # This will be updated once 'df' and 'label' column are processed

# --- BERT ---
bert_model_name = 'bert-base-uncased'
bert_tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
bert_model = AutoModelForSequenceClassification.from_pretrained(bert_model_name, num_labels=num_unique_labels) # Use dynamic num_labels
bert_model.to(device)
print(f"\n{bert_model_name} loaded.")

# --- DistilBERT ---
distilbert_model_name = 'distilbert-base-uncased'
distilbert_tokenizer = AutoTokenizer.from_pretrained(distilbert_model_name)
distilbert_model = AutoModelForSequenceClassification.from_pretrained(distilbert_model_name, num_labels=num_unique_labels)
distilbert_model.to(device)
print(f"\n{distilbert_model_name} loaded.")

# --- RoBERTa ---
roberta_model_name = 'roberta-base'
roberta_tokenizer = AutoTokenizer.from_pretrained(roberta_model_name)
roberta_model = AutoModelForSequenceClassification.from_pretrained(roberta_model_name, num_labels=num_unique_labels)
roberta_model.to(device)
print(f"\n{roberta_model_name} loaded.")

Using device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



bert-base-uncased loaded.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



distilbert-base-uncased loaded.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



roberta-base loaded.


In [ ]:
if df is not None:
    # Concatenate 'Premise' and 'Hypothesis' into a single 'text' column
    # Fill any NaN values with an empty string before concatenation
    df['text'] = df['Premise'].fillna('') + ' ' + df['Hypothesis'].fillna('')

    # Display some examples of the combined text
    print("\nExamples of combined text:")
    for i in range(min(5, len(df))):
        print(f"- {df['text'].iloc[i][:150]}...") # Print first 150 characters

    # Inspect the 'Relationship' column for unique labels
    if 'Relationship' in df.columns:
        print("\nUnique labels in 'Relationship' column:")
        print(df['Relationship'].value_counts())

        # Convert 'Relationship' to categorical type and then to numerical codes
        # This will assign a unique integer to each unique string label
        df['label'] = pd.Categorical(df['Relationship']).codes

        # Get the mapping from category name to code
        category_to_code = {category: code for code, category in enumerate(pd.Categorical(df['Relationship']).categories)}
        print("\nCreated 'label' column with mapping:")
        print(category_to_code)
        print(df['label'].value_counts())

        # Update num_unique_labels for model initialization (if df is available globally)
        # Note: This requires rerunning the model initialization cell after this cell.
        global num_unique_labels
        num_unique_labels = len(pd.Categorical(df['Relationship']).categories)
        print(f"\nDetected {num_unique_labels} unique labels. Please ensure you rerun the model initialization cell to update `num_labels`.")

    else:
        print("\nWarning: 'Relationship' column not found. You will need to define your 'label' column manually for classification.")
        df['label'] = None
else:
    print("DataFrame 'df' is not available. Please ensure the Excel file was loaded correctly.")


Examples of combined text:
- The quantum processor developed by QTech Corp can perform over a million operations per second. The quantum processor from QTech Corp is limited to ju...
- The new privacy update encrypts all user data end-to-end using a zero-knowledge protocol. The new privacy update fails to encrypt any user data....
- The AI model was trained using a dataset consisting exclusively of financial news articles. The AI model was trained using diverse datasets including ...
- Tesla’s autopilot system was launched in 2015 and has undergone regular updates since then. Tesla’s autopilot was introduced in 2022 after years of de...
- The cloud server in Frankfurt ensures data is stored within the EU for regulatory compliance. The cloud server is located in Singapore to reduce laten...

Unique labels in 'Relationship' column:
Relationship
Contradictory    1028
Entailing         904
Neutral           648
Name: count, dtype: int64

Created 'label' column with mapping:
{'Contradictor

### 4. Tokenization

Now we'll tokenize the 'text' column using each model's respective tokenizer. We'll also split the data into training and validation sets.

In [ ]:
from sklearn.model_selection import train_test_split

# Ensure num_unique_labels is correctly set based on the 'label' column
# This check is crucial if the user reran the preprocessing cell separately.
if 'df' in globals() and df is not None and 'label' in df.columns:
    current_num_labels = len(df['label'].unique())
    print(f"Detected {current_num_labels} unique labels in the dataset.")
    # Optionally, re-initialize models if num_unique_labels has changed from last run of cell 853a0a79
    # This part can be more complex if models need full re-init. For simplicity, we assume
    # the user has re-run the model initialization cell if prompted.
else:
    print("Warning: 'df' or 'label' column not found. Tokenization may fail or use default num_labels (3).")
    current_num_labels = num_unique_labels # Fallback to global placeholder if df is not ready

if df is not None and 'text' in df.columns and 'label' in df.columns and df['label'].notna().any():
    # Drop rows where text or label is missing for training
    df_cleaned = df.dropna(subset=['text', 'label']).copy()

    # Convert labels to integers
    df_cleaned['label'] = df_cleaned['label'].astype(int)

    # Split data into training and validation sets
    train_texts, val_texts, train_labels, val_labels = train_test_split(
        df_cleaned['text'].tolist(),
        df_cleaned['label'].tolist(),
        test_size=0.2, # 20% for validation
        random_state=42,
        stratify=df_cleaned['label'].tolist() # Ensure balanced classes in splits
    )

    print(f"\nTraining samples: {len(train_texts)}")
    print(f"Validation samples: {len(val_texts)}")

    # Tokenize data for each model
    def tokenize_data(tokenizer, texts):
        return tokenizer(texts, padding=True, truncation=True, return_tensors='pt')

    print("\nTokenizing data for BERT...")
    train_encodings_bert = tokenize_data(bert_tokenizer, train_texts)
    val_encodings_bert = tokenize_data(bert_tokenizer, val_texts)

    print("Tokenizing data for DistilBERT...")
    train_encodings_distilbert = tokenize_data(distilbert_tokenizer, train_texts)
    val_encodings_distilbert = tokenize_data(distilbert_tokenizer, val_texts)

    print("Tokenizing data for RoBERTa...")
    train_encodings_roberta = tokenize_data(roberta_tokenizer, train_texts)
    val_encodings_roberta = tokenize_data(roberta_tokenizer, val_texts)

    print("Tokenization complete.")
else:
    print("Cannot proceed with tokenization: DataFrame 'df' is not available, or 'text'/'label' columns are missing/empty. Please ensure the Excel file was loaded and preprocessed correctly.")


Detected 3 unique labels in the dataset.

Training samples: 2064
Validation samples: 516

Tokenizing data for BERT...
Tokenizing data for DistilBERT...
Tokenizing data for RoBERTa...
Tokenization complete.


### 5. Create PyTorch Datasets and DataLoaders

To train and evaluate our models, we need to convert our tokenized data and labels into PyTorch `Dataset` objects and then wrap them with `DataLoader`s. This helps in efficient batching and shuffling of data.

In [ ]:
from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx].clone().detach() for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

if 'train_encodings_bert' in locals():
    # Create datasets
    train_dataset_bert = TextDataset(train_encodings_bert, train_labels)
    val_dataset_bert = TextDataset(val_encodings_bert, val_labels)

    train_dataset_distilbert = TextDataset(train_encodings_distilbert, train_labels)
    val_dataset_distilbert = TextDataset(val_encodings_distilbert, val_labels)

    train_dataset_roberta = TextDataset(train_encodings_roberta, train_labels)
    val_dataset_roberta = TextDataset(val_encodings_roberta, val_labels)

    # Create DataLoaders
    batch_size = 16 # You can adjust this based on your GPU memory and preference

    train_loader_bert = DataLoader(train_dataset_bert, batch_size=batch_size, shuffle=True)
    val_loader_bert = DataLoader(val_dataset_bert, batch_size=batch_size, shuffle=False)

    train_loader_distilbert = DataLoader(train_dataset_distilbert, batch_size=batch_size, shuffle=True)
    val_loader_distilbert = DataLoader(val_dataset_distilbert, batch_size=batch_size, shuffle=False)

    train_loader_roberta = DataLoader(train_dataset_roberta, batch_size=batch_size, shuffle=True)
    val_loader_roberta = DataLoader(val_dataset_roberta, batch_size=batch_size, shuffle=False)

    print("PyTorch Datasets and DataLoaders created for all models.")
else:
    print("Cannot create Datasets/DataLoaders: Tokenized data is not available. Please ensure previous steps ran successfully.")

PyTorch Datasets and DataLoaders created for all models.


### 6. Training and Evaluation Strategy

Now that the data is prepared, we can proceed with training and evaluating the models. There are several approaches you can take:

**Option A: Fine-tuning with `Trainer` API (Recommended)**

The `transformers` library provides a high-level `Trainer` API that simplifies the fine-tuning process significantly. This is generally the easiest and most robust way to fine-tune Hugging Face models.

**Option B: Manual Training Loop**

If you need more control over the training process, you can write a custom PyTorch training loop. This involves iterating through `DataLoader`s, performing forward passes, calculating loss, backpropagating, and updating weights.

**Option C: Zero-shot Classification (if applicable)**

If your task is suitable for zero-shot classification (i.e., classifying text into categories without explicit training data for those categories), you could use a zero-shot classification pipeline. However, since you have an Excel dataset, fine-tuning is usually more appropriate for better performance on your specific data.

---

**For this walkthrough, I will proceed with Option A (Fine-tuning with `Trainer` API) as it's the most common and efficient way to fine-tune these models.** We will define a training function that can be reused for each model.

### 7. Fine-tuning Models with `Trainer` API

We will now fine-tune each of the models using the `Trainer` API from the `transformers` library. This involves:

1.  **Defining `TrainingArguments`**: Specifies training hyperparameters.
2.  **Defining `compute_metrics` function**: Calculates evaluation metrics (e.g., accuracy, F1-score).
3.  **Initializing `Trainer`**: Combines the model, arguments, datasets, and metrics.
4.  **Training the model**: Calling the `train()` method.
5.  **Evaluating the model**: Calling the `evaluate()` method.

In [ ]:
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import numpy as np
import pandas as pd # Ensure pandas is imported

# Define label_names globally, based on how labels were encoded in the preprocessing step.
# This assumes df is available and 'Relationship' column exists from previous steps.
if 'df' in globals() and df is not None and 'Relationship' in df.columns:
    global label_names
    label_names = pd.Categorical(df['Relationship']).categories.tolist()
else:
    # Fallback if df is not available or 'Relationship' column is missing
    # In this case, we rely on num_unique_labels set earlier (default 3).
    label_names = [f"class_{i}" for i in range(num_unique_labels)]
    print(f"Warning: 'label_names' could not be determined from df. Using generic labels: {label_names}")

def compute_metrics(p):
    predictions = np.argmax(p.predictions, axis=1);
    labels = p.label_ids

    metrics = {
        'accuracy': accuracy_score(labels, predictions),
        'f1_weighted': f1_score(labels, predictions, average='weighted', zero_division=0),
        'precision_weighted': precision_score(labels, predictions, average='weighted', zero_division=0),
        'recall_weighted': recall_score(labels, predictions, average='weighted', zero_division=0)
    }

    # Calculate per-class F1 scores
    # The 'labels' parameter in f1_score ensures scores are computed for all possible classes (0, 1, ..., num_unique_labels-1)
    # and that the output array order corresponds to these integer labels.
    all_possible_labels_as_int = list(range(num_unique_labels))
    f1_per_class = f1_score(labels, predictions, average=None, labels=all_possible_labels_as_int, zero_division=0)

    for i, f1_score_val in enumerate(f1_per_class):
        # Map the integer index 'i' back to its meaningful label name
        if i < len(label_names):
            metrics[f'f1_{label_names[i]}'] = f1_score_val
        else:
            metrics[f'f1_class_{i}'] = f1_score_val # Fallback for unexpected scenarios

    return metrics

# Check if datasets and loaders were successfully created
if 'train_dataset_bert' not in locals():
    print("Training cannot proceed: Datasets/DataLoaders not available. Please ensure previous steps ran successfully.")
else:
    # --- Training Arguments ---
    # You can customize these arguments based on your needs and resources
    training_args = TrainingArguments(
        output_dir='./results',
        num_train_epochs=3,               # Total number of training epochs
        per_device_train_batch_size=16,  # Batch size per GPU/CPU for training
        per_device_eval_batch_size=16,   # Batch size per GPU/CPU for evaluation
        warmup_steps=500,                 # Number of warmup steps for learning rate scheduler
        weight_decay=0.01,                # Strength of weight decay
        logging_steps=100,                # Log every x updates steps
        eval_strategy="epoch",      # Evaluate every epoch
        save_strategy="epoch",            # Save model every epoch
        load_best_model_at_end=True,      # Load the best model after training
        metric_for_best_model="f1_weighted",       # Metric to use to compare models (using weighted F1)
        greater_is_better=True,
        report_to="none"                  # Don't report to any online service like Weights & Biases
    )

    models_to_train = {
        "BERT": {"model": bert_model, "tokenizer": bert_tokenizer, "train_dataset": train_dataset_bert, "eval_dataset": val_dataset_bert},
        "DistilBERT": {"model": distilbert_model, "tokenizer": distilbert_tokenizer, "train_dataset": train_dataset_distilbert, "eval_dataset": val_dataset_distilbert},
        "RoBERTa": {"model": roberta_model, "tokenizer": roberta_tokenizer, "train_dataset": train_dataset_roberta, "eval_dataset": val_dataset_roberta}
    }

    evaluation_results = {}

    for model_name, data in models_to_train.items():
        print(f"\n--- Training and Evaluating {model_name} ---")
        trainer = Trainer(
            model=data["model"],
            args=training_args,
            train_dataset=data["train_dataset"],
            eval_dataset=data["eval_dataset"],
            compute_metrics=compute_metrics
        )

        # Train the model
        trainer.train()

        # Evaluate the model
        eval_metrics = trainer.evaluate()
        evaluation_results[model_name] = eval_metrics
        print(f"Evaluation results for {model_name}: {eval_metrics}")

    print("\n--- All Models Trained and Evaluated ---")
    print("Summary of Evaluation Results:")
    for model_name, metrics in evaluation_results.items():
        print(f"\n{model_name}:")
        for metric, value in metrics.items():
            print(f"  {metric}: {value:.4f}")


--- Training and Evaluating BERT ---


Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted,Precision Weighted,Recall Weighted,F1 Contradictory,F1 Entailing,F1 Neutral
1,1.069504,0.817062,0.645349,0.589592,0.665494,0.645349,0.774067,0.311688,0.684932
2,0.772823,0.613178,0.750000,0.739354,0.773991,0.750000,0.816514,0.634146,0.763754
3,0.472010,0.534253,0.825581,0.824516,0.825048,0.825581,0.844869,0.778098,0.857143


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,F1 Weighted,Precision Weighted,Recall Weighted,F1 Contradictory,F1 Entailing,F1 Neutral
0.357229,0.367067,3,0.877907,0.877435,0.881056,0.877907,0.900943,0.861357,0.862454


Evaluation results for RoBERTa: {'eval_loss': 0.36706671118736267, 'eval_accuracy': 0.877906976744186, 'eval_f1_weighted': 0.8774349804624297, 'eval_precision_weighted': 0.8810564290209039, 'eval_recall_weighted': 0.877906976744186, 'eval_f1_Contradictory': 0.9009433962264151, 'eval_f1_Entailing': 0.8613569321533924, 'eval_f1_Neutral': 0.862453531598513}

--- All Models Trained and Evaluated ---
Summary of Evaluation Results:

BERT:
  eval_loss: 0.5343
  eval_accuracy: 0.8256
  eval_f1_weighted: 0.8245
  eval_precision_weighted: 0.8250
  eval_recall_weighted: 0.8256
  eval_f1_Contradictory: 0.8449
  eval_f1_Entailing: 0.7781
  eval_f1_Neutral: 0.8571

DistilBERT:
  eval_loss: 0.6000
  eval_accuracy: 0.7597
  eval_f1_weighted: 0.7490
  eval_precision_weighted: 0.7766
  eval_recall_weighted: 0.7597
  eval_f1_Contradictory: 0.8303
  eval_f1_Entailing: 0.6370
  eval_f1_Neutral: 0.7763

RoBERTa:
  eval_loss: 0.3671
  eval_accuracy: 0.8779
  eval_f1_weighted: 0.8774
  eval_precision_weighted

### 8. Conclusion and Comparison

Based on the evaluation results above, you can compare the performance of BERT, DistilBERT, and RoBERTa on your specific dataset. Key metrics to consider are:

*   **Accuracy**: Overall correctness of predictions.
*   **F1-Score**: Harmonic mean of precision and recall, good for imbalanced datasets.
*   **Precision**: Proportion of positive identifications that were actually correct.
*   **Recall**: Proportion of actual positives that were identified correctly.

DistilBERT is generally faster and smaller than BERT, but may have slightly lower performance. RoBERTa often outperforms BERT due to its training methodology but is typically similar in size and speed to BERT. The 'best' model will depend on your specific task's requirements for performance, speed, and resource usage.

In [ ]:
import pandas as pd # Ensure pandas is imported

print("\n--- Detailed F1 Scores per Class ---")

if 'evaluation_results' in globals():
    # Attempt to define label_names if not already in global scope, for independent execution.
    if 'label_names' not in globals() and 'df' in globals() and df is not None and 'Relationship' in df.columns:
        label_names = pd.Categorical(df['Relationship']).categories.tolist()
    elif 'label_names' not in globals():
        label_names = [] # Initialize to empty list if df not available

    for model_name, metrics in evaluation_results.items():
        print(f"\n--- {model_name} ---")
        found_per_class_f1 = False

        if label_names:
            # Iterate through known label names to print their F1 scores
            for label_name in label_names:
                key = f'f1_{label_name}'
                if key in metrics:
                    print(f"  F1 Score ({label_name}): {metrics[key]:.4f}")
                    found_per_class_f1 = True
        else: # Fallback if label_names couldn't be determined dynamically
            # Try to infer labels from the metric keys themselves (e.g., f1_class_0, f1_class_1)
            inferred_f1_keys = sorted([k for k in metrics if k.startswith('f1_class_')])
            if inferred_f1_keys:
                print("  (Using inferred class names as specific label names not found):")
                for key in inferred_f1_keys:
                    # Format key for better readability, e.g., 'f1_class_0' -> 'F1 Score (class_0)'
                    display_name = key.replace('f1_', 'F1 Score (') + ')'
                    print(f"  {display_name}: {metrics[key]:.4f}")
                found_per_class_f1 = True

        if not found_per_class_f1:
            print("  Per-class F1 scores not found. Ensure the evaluation cell (step 7) was run after the update.")
            # Also print weighted F1 for context if per-class not found
            if 'f1_weighted' in metrics:
                print(f"  Weighted F1 Score: {metrics['f1_weighted']:.4f}")
            elif 'eval_f1' in metrics: # Backward compatibility with original output if not updated
                print(f"  Weighted F1 Score (Legacy): {metrics['eval_f1']:.4f}")
else:
    print("Evaluation results not available. Please run the training and evaluation cell (step 7) first.")